In [1]:
# Make helpers.py importable, whatever folder Jupyter started the kernel in
import sys
from pathlib import Path

_candidates = [Path.cwd(), Path.cwd() / "code",
               *Path.cwd().glob("*/code"), *Path.cwd().glob("*/*/code")]
CODE = next(p for p in _candidates if (p / "helpers.py").is_file())
sys.path.insert(0, str(CODE))

In [2]:
# Generated with Claude (Anthropic): clean the OCHA FTS data, add country
# characteristics (World Bank, ReliefWeb) and build the analysis table.
import numpy as np
import pandas as pd

from helpers import find_root, read_worldbank

ROOT = find_root()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(exist_ok=True)

FIRST_YEAR, LAST_YEAR = 2000, 2025
UNDERFUNDED_THRESHOLD = 0.5   # underfunded = received less than 50% of requirements
                              # (threshold of the Humanitarian Funding Forecast index)
MIN_ROWS_PER_SECTOR = 30      # sectors with fewer usable rows are dropped

steps = []  # number of rows after each cleaning step, for the paper


def log(step, df):
    steps.append({"step": step, "rows": len(df)})
    print(f"{step}: {len(df)} rows")


print("Repository folder:", ROOT)

Repository folder: /files/project/humanitarian-underfunding


In [3]:
fts = pd.read_csv(RAW / "fts_globalcluster.csv")
log("1. Raw OCHA file", fts)

fts = fts[fts["year"].between(FIRST_YEAR, LAST_YEAR)]
log(f"2. Years {FIRST_YEAR}-{LAST_YEAR} (2026 is incomplete)", fts)

catch_all = ["Not specified", "Multiple clusters/sectors (shared)"]

# Share of each appeal's funding not attributed to a sector (for a robustness check)
appeal_funding = fts.groupby(["id", "year"])["funding"].sum()
unattributed = fts[fts["cluster"].isin(catch_all)].groupby(["id", "year"])["funding"].sum()
unattributed_share = (unattributed / appeal_funding).fillna(0).rename("unattributed_share")

fts = fts[~fts["cluster"].isin(catch_all)]
log("3. Without funding not attributed to a sector", fts)

fts = fts[fts["requirements"] > 0]
log("4. Without missing or zero amount requested", fts)

fts = fts[fts["cluster"] != "Other"]
log("5. Without the 'Other' sector", fts)

sector_size = fts["cluster"].value_counts()
rare = sector_size[sector_size < MIN_ROWS_PER_SECTOR].index.tolist()
print("Rare sectors dropped:", rare)
fts = fts[~fts["cluster"].isin(rare)]
log(f"6. Without sectors with fewer than {MIN_ROWS_PER_SECTOR} rows", fts)

1. Raw OCHA file: 10701 rows
2. Years 2000-2025 (2026 is incomplete): 10164 rows
3. Without funding not attributed to a sector: 8594 rows
4. Without missing or zero amount requested: 7824 rows
5. Without the 'Other' sector: 7816 rows
Rare sectors dropped: ['Protection - Human Trafficking & Smuggling']
6. Without sectors with fewer than 30 rows: 7807 rows


In [4]:
# FTS relies on voluntary reporting: keep track of rows with no funding reported
fts["no_funding_reported"] = (fts["funding"].isna() | (fts["funding"] == 0)).astype(int)
fts["funding"] = fts["funding"].fillna(0)   # no funding recorded = nothing received
fts["funding_rate"] = fts["funding"] / fts["requirements"]
fts["underfunded"] = (fts["funding_rate"] < UNDERFUNDED_THRESHOLD).astype(int)
fts["log_requirements"] = np.log(fts["requirements"])

fts = fts.merge(unattributed_share.reset_index(), on=["id", "year"], how="left")

print("Share underfunded:", round(fts["underfunded"].mean(), 3))
print("Rows funded above 100%:", (fts["funding_rate"] > 1).sum())
print("Rows funded above 1000%:", (fts["funding_rate"] > 10).sum())
print("Rows with no funding reported:", fts["no_funding_reported"].sum())

Share underfunded: 0.634
Rows funded above 100%: 616
Rows funded above 1000%: 21
Rows with no funding reported: 863


In [5]:
wb = pd.concat([read_worldbank(RAW / "wgi.csv"),
                read_worldbank(RAW / "wdi_gdppc.csv")])
country = wb.pivot_table(index=["countryCode", "year"], columns="series",
                         values="value").reset_index()
country = country.rename(columns={"GOV_WGI_GE.EST": "gov_effectiveness",
                                  "GOV_WGI_CC.EST": "control_corruption",
                                  "NY.GDP.PCAP.KD": "gdp_pc"})
country.columns.name = None
print(country.columns.tolist())

# WGI were published every two years until 2002 (no 1997, 1999, 2001):
# carry the previous value forward within each country
country = country.sort_values(["countryCode", "year"])
all_years = pd.MultiIndex.from_product(
    [country["countryCode"].unique(), range(country["year"].min(), LAST_YEAR + 1)],
    names=["countryCode", "year"])
country = country.set_index(["countryCode", "year"]).reindex(all_years).reset_index()
for col in ["gov_effectiveness", "control_corruption"]:
    country[col] = country.groupby("countryCode")[col].ffill(limit=1)

reliefweb = pd.read_csv(RAW / "reliefweb_counts.csv")
country = country.merge(reliefweb, on=["countryCode", "year"], how="left")

['countryCode', 'year', 'control_corruption', 'gov_effectiveness', 'gdp_pc']


In [6]:
# Lag: an appeal of year t is matched with country characteristics of year t-1
# (governance indicators are published with about a one-year delay)
lagged = country.copy()
lagged["year"] = lagged["year"] + 1
lagged = lagged.rename(columns={c: c + "_lag" for c in
                                ["gov_effectiveness", "control_corruption",
                                 "gdp_pc", "reports"]})

df = fts.merge(lagged, on=["countryCode", "year"], how="left")

# Same-year characteristics, only for a robustness check
df = df.merge(country, on=["countryCode", "year"], how="left")

log("7. After merging country characteristics (t-1)", df)

df["log_gdp_pc_lag"] = np.log(df["gdp_pc_lag"])
df["log_reports_lag"] = np.log1p(df["reports_lag"])   # log(1 + reports): 0 reports is allowed

print(df[["gdp_pc_lag", "gov_effectiveness_lag", "control_corruption_lag",
          "reports_lag"]].isna().sum())
print(df.loc[df["gdp_pc_lag"].isna(), "countryCode"].value_counts())

7. After merging country characteristics (t-1): 7807 rows
gdp_pc_lag                373
gov_effectiveness_lag      21
control_corruption_lag     10
reports_lag                 0
dtype: int64
countryCode
SSD    128
YEM     86
PRK     74
SYR     43
DJI     42
Name: count, dtype: int64


In [7]:
main_vars = ["log_gdp_pc_lag", "gov_effectiveness_lag", "log_reports_lag"]
missing = df[df[main_vars].isna().any(axis=1)]
print("Rows with a missing country characteristic, by country:")
print(missing["countryCode"].value_counts())

df_full = df.copy()   # before dropping: used by robustness checks

df = df.dropna(subset=main_vars)
log("8. Without missing country characteristics", df)

Rows with a missing country characteristic, by country:
countryCode
SSD    139
YEM     86
PRK     74
SYR     43
DJI     42
TLS     10
Name: count, dtype: int64
8. Without missing country characteristics: 7413 rows


In [8]:
print("Correlation government effectiveness / control of corruption:",
      round(df[["gov_effectiveness_lag", "control_corruption_lag"]].corr().iloc[0, 1], 2))
print("Countries:", df["countryCode"].nunique(), "| Appeals:", df["id"].nunique(),
      "| Sectors:", df["cluster"].nunique())
print(df.groupby("cluster")["underfunded"].agg(["mean", "size"]).round(2)
        .sort_values("mean"))

Correlation government effectiveness / control of corruption: 0.72
Countries: 83 | Appeals: 740 | Sectors: 19
                                         mean  size
cluster                                            
Food Security                            0.36   691
Logistics                                0.39   265
Coordination and support services        0.42   650
Multi-sector                             0.54   351
Multipurpose Cash                        0.54    41
Protection                               0.63   648
Emergency Telecommunications             0.65   117
Health                                   0.66   727
Nutrition                                0.67   303
Protection - Mine Action                 0.68   217
Water Sanitation Hygiene                 0.69   710
Agriculture                              0.71   404
Education                                0.74   669
Emergency Shelter and NFI                0.75   548
Protection - Child Protection            0.78   221
Protec

In [9]:
columns = ["countryCode", "id", "code", "name", "year", "cluster",
           "requirements", "funding", "funding_rate", "underfunded",
           "no_funding_reported", "log_requirements",
           "gdp_pc_lag", "log_gdp_pc_lag",
           "gov_effectiveness_lag", "control_corruption_lag",
           "reports_lag", "log_reports_lag"]
df[columns].to_csv(PROCESSED / "analysis_table.csv", index=False)

full_columns = columns + ["unattributed_share", "gdp_pc", "gov_effectiveness",
                          "control_corruption", "reports"]
df_full[full_columns].to_csv(PROCESSED / "analysis_table_full.csv", index=False)
print("Saved", PROCESSED / "analysis_table_full.csv", df_full[full_columns].shape)

pd.DataFrame(steps).to_csv(PROCESSED / "cleaning_steps.csv", index=False)
print("Saved", PROCESSED / "analysis_table.csv", df[columns].shape)
pd.DataFrame(steps)

Saved /files/project/humanitarian-underfunding/data/processed/analysis_table_full.csv (7807, 23)
Saved /files/project/humanitarian-underfunding/data/processed/analysis_table.csv (7413, 18)


,step,rows
0,1. Raw OCHA file,10701
1,2. Years 2000-2025 (2026 is incomplete),10164
2,3. Without funding not attributed to a sector,8594
3,4. Without missing or zero amount requested,7824
4,5. Without the 'Other' sector,7816
5,6. Without sectors with fewer than 30 rows,7807
6,7. After merging country characteristics (t-1),7807
7,8. Without missing country characteristics,7413
